In [ ]:
import pandas as pd
import numpy as np
from scipy.special import erf
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import warnings

warnings.filterwarnings('ignore')

# --- 1. Load and Advanced Preprocessing ---
def load_and_preprocess():
    train = pd.read_csv('/kaggle/input/datasets/sasidhar412/widsdataset/train.csv')
    test = pd.read_csv('/kaggle/input/datasets/sasidhar412/widsdataset/test.csv')
    
    def transform(df):
        df = df.copy()
        df['closing_speed_m_per_h'] = df['closing_speed_m_per_h'].fillna(df['closing_speed_m_per_h'].median())
        df['radial_growth_rate_m_per_h'] = df['radial_growth_rate_m_per_h'].fillna(df['radial_growth_rate_m_per_h'].median())
        df['area_growth_rel_0_5h'] = df['area_growth_rel_0_5h'].clip(upper=df['area_growth_rel_0_5h'].quantile(0.99))
        df['persistence'] = np.log1p(df['num_perimeters_0_5h'])
        df['dist'] = (df['dist_min_ci_0_5h'] - 5000).clip(lower=10)
        
        # Core Physics Meta-Feature
        df['v_stable'] = (df['closing_speed_m_per_h'] + df['radial_growth_rate_m_per_h']).clip(lower=1.0) * \
                         (df['alignment_abs']**1.5) * df['persistence']
        
        # Interaction Features
        df['eta_kinetic'] = df['dist'] / df['v_stable'].clip(lower=0.1)
        df['density_metric'] = df['area_first_ha'] / (df['dist'] + 1)
        df['speed_alignment'] = df['closing_speed_m_per_h'] * df['alignment_abs']
        return df
        
    return transform(train), transform(test)

train_df, test_df = load_and_preprocess()

# --- 2. Physics & Metric Engines ---
def get_physics_probs(df, train_ref):
    S_C, P_W, SIGMA, BIAS = 1.83455702, 0.00975066, -1.11494169, -1.17324153
    HORIZONS = [12, 24, 48, 72]
    LK_FAST, LK_SLOW = [12.0, 15.0, 16.0, 16.0], [8.0, 10.0, 8.0, 12.0]
    QUANTILES = [0.85, 0.65, 0.65, 0.65]
    
    probs = np.zeros((len(df), 4))
    t_kinetic = df['dist'] / df['v_stable'].clip(lower=1.0)
    mu_log = np.log((t_kinetic * S_C).clip(0.1, 1000)) - P_W * np.log1p(df['area_growth_rel_0_5h'])
    sig = np.exp(SIGMA).clip(0.1, 5)
    
    for i, t in enumerate(HORIZONS):
        v_thresh = train_ref['v_stable'].quantile(QUANTILES[i])
        is_fast = (df['v_stable'] > v_thresh).values
        z = (np.log(t) - mu_log) / (sig * np.sqrt(2))
        p_arr = 0.5 * (1 + erf(z))
        lk_vec = np.where(is_fast, LK_FAST[i], LK_SLOW[i])
        probs[:, i] = 1.0 / (1.0 + np.exp(-(lk_vec * (p_arr - 0.5) + BIAS)))
    return probs

def calculate_metric(probs, times, events):
    b_weights = [0.20, 0.40, 0.40]
    b_total = 0
    for i, h in enumerate([12, 24, 48, 72]):
        if i == 0: continue
        mask = (events == 1) | (times >= h)
        y_true = ((times <= h) & (events == 1)).astype(int)[mask]
        b_total += b_weights[i-1] * brier_score_loss(y_true, probs[mask, i])
    c_score = roc_auc_score(((times <= 72) & (events == 1)).astype(int), probs[:, 3])
    return 0.10 * c_score + 0.90 * (1 - b_total)

# --- 3. Triple Ensemble CV ---
X_features = ['dist', 'v_stable', 'eta_kinetic', 'alignment_abs', 'area_first_ha', 'density_metric', 'speed_alignment']
y_target = ((train_df['time_to_hit_hours'] <= 72) & (train_df['event'] == 1)).astype(int)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = []

# Triple Blend Weights
W_PHYS = 0.85
W_XGB = 0.10
W_RF = 0.05
SQUISH = 1.20

print(f"--- Running Triple-Threat CV (Phys: {W_PHYS}, XGB: {W_XGB}, RF: {W_RF}) ---")

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, y_target)):
    df_tr, df_val = train_df.iloc[train_idx], train_df.iloc[val_idx]
    y_time_val, y_event_val = df_val['time_to_hit_hours'].values, df_val['event'].values
    
    # 1. Physics
    p_phys = get_physics_probs(df_val, df_tr)
    
    # 2. XGB & RF Predictions
    p_xgb, p_rf = np.zeros_like(p_phys), np.zeros_like(p_phys)
    for i, h in enumerate([12, 24, 48, 72]):
        y_tr_h = ((df_tr['time_to_hit_hours'] <= h) & (df_tr['event'] == 1)).astype(int)
        
        # XGB Sniper
        xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.03, random_state=42, verbosity=0)
        xgb.fit(df_tr[X_features], y_tr_h)
        p_xgb[:, i] = xgb.predict_proba(df_val[X_features])[:, 1]
        
        # RF Generalist
        rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
        rf.fit(df_tr[X_features], y_tr_h)
        p_rf[:, i] = rf.predict_proba(df_val[X_features])[:, 1]
    
    # 3. Ensemble Blend
    blend_val = (W_PHYS * p_phys) + (W_XGB * p_xgb) + (W_RF * p_rf)
    
    # 4. Calibration & Penalty
    final_val = np.power(blend_val, SQUISH)
    final_val[df_val['num_perimeters_0_5h'] == 0] *= 0.95
    
    fold_score = calculate_metric(final_val, y_time_val, y_event_val)
    cv_scores.append(fold_score)
    print(f"Fold {fold+1}: {fold_score:.6f}")

print(f"\nTriple-Threat Mean CV: {np.mean(cv_scores):.6f}")

# --- 4. Submission Inference ---
print("\n--- Generating Triple-Threat Master Submission ---")
p_phys_test = get_physics_probs(test_df, train_df)
p_xgb_test, p_rf_test = np.zeros_like(p_phys_test), np.zeros_like(p_phys_test)

for i, h in enumerate([12, 24, 48, 72]):
    y_tr_h = ((train_df['time_to_hit_hours'] <= h) & (train_df['event'] == 1)).astype(int)
    
    xgb = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.03, random_state=42, verbosity=0)
    xgb.fit(train_df[X_features], y_tr_h)
    p_xgb_test[:, i] = xgb.predict_proba(test_df[X_features])[:, 1]
    
    rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
    rf.fit(train_df[X_features], y_tr_h)
    p_rf_test[:, i] = rf.predict_proba(test_df[X_features])[:, 1]

# Final Blend
final_probs = np.power((W_PHYS * p_phys_test) + (W_XGB * p_xgb_test) + (W_RF * p_rf_test), SQUISH)
final_probs[test_df['num_perimeters_0_5h'] == 0] *= 0.95

submission = pd.DataFrame({'event_id': test_df['event_id'], 'prob_12h': final_probs[:, 0], 'prob_24h': final_probs[:, 1], 'prob_48h': final_probs[:, 2], 'prob_72h': final_probs[:, 3]})
submission[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']] = np.sort(submission[['prob_12h', 'prob_24h', 'prob_48h', 'prob_72h']].values, axis=1)
submission.to_csv('submission231.csv', index=False)
print("Saved: submission_triple_threat.csv. This is the 0.991 contender.")